<a href="https://colab.research.google.com/github/csabiu/Cosmology_Course/blob/main/practical/zeldovich_cosmic_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Zel'dovich Approximation and the Cosmic Web

---

In this practical you will:
1. **Generate a Gaussian random field** from a power spectrum $P(k)$
2. **Compute displacement fields** via FFT from the Poisson equation
3. **Apply the Zel'dovich mapping** $\mathbf{x} = \mathbf{q} + D(t)\,\boldsymbol{\Psi}(\mathbf{q})$
4. **Watch pancakes, filaments, and nodes form** as the growth factor $D$ increases
5. **Measure the power spectrum** of the evolved field and compare to the linear input

**Parts 1--4** are guided tutorials. **Part 5** contains one exercise.

**Prerequisites:** Lecture 5 (linear perturbation theory), Lecture 6 (nonlinear structure formation, Zel'dovich approximation).

**Estimated time:** 1.5 hours.

## Part 0 --- Setup

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'axes.grid': False,
})

# Planck 2018 fiducial cosmology
Om0 = 0.3153
OL0 = 1.0 - Om0
h = 0.6736
Om_h2 = Om0 * h**2
ns_fid = 0.9649

print(f"Cosmology: Om = {Om0}, OL = {OL0:.4f}, h = {h}")

## Part 1 --- The Initial Gaussian Random Field

Inflation produces a Gaussian random field $\delta(\mathbf{x})$ whose statistical properties are fully described by the power spectrum $P(k)$. In 2D we generate this on an $N \times N$ grid using the FFT:

1. Draw random Fourier amplitudes $\tilde\delta(\mathbf{k})$ with variance $\propto P(k)$
2. Inverse-FFT to get the real-space field $\delta(\mathbf{x})$

We use the **BBKS matter power spectrum** $P(k) = k^{n_s}\,T^2(k)$, where $T(k)$ is the transfer function from Lecture 5.  The turnover at $k_{\rm eq}$ suppresses small-scale power and is essential for producing the characteristic cosmic web morphology.

In [ ]:
# ============================================================
# Tutorial: Helper functions
# ============================================================

# --- Growth factor D(z) for flat LCDM ---
def growth_factor(z, Om0=Om0, OL0=OL0):
    """Linear growth factor D(z), normalised to D(0) = 1.
    Uses the Heath integral: D(a) propto E(a) * integral_0^a da'/(a'E(a'))^3.
    """
    def E(a):
        return np.sqrt(Om0 / a**3 + OL0)
    def integrand(a):
        return 1.0 / (a * E(a))**3
    a = 1.0 / (1.0 + z)
    D_a = E(a) * quad(integrand, 1e-8, a)[0]
    D_0 = E(1.0) * quad(integrand, 1e-8, 1.0)[0]
    return D_a / D_0

# Print D(z) at key redshifts
print("Growth factor D(z) [normalised to D(0) = 1]:")
for z in [0, 1, 3, 10, 50]:
    print(f"  z = {z:2d}:  D = {growth_factor(z):.4f}")

# --- BBKS transfer function ---
def transfer_BBKS(k, Om_h2=Om_h2):
    """BBKS transfer function T(k). k in Mpc^{-1}."""
    q = k / Om_h2
    T = (np.log(1 + 2.34*q) / (2.34*q)) * \
        (1 + 3.89*q + (16.1*q)**2 + (5.46*q)**3 + (6.71*q)**4)**(-0.25)
    return np.where(k == 0, 1.0, T)

# --- Power spectrum ---
def power_spectrum_2d(kx, ky, ns=ns_fid, Om_h2=Om_h2):
    """2D matter power spectrum P(k) = k^ns * T^2(k) with BBKS transfer function."""
    k = np.sqrt(kx**2 + ky**2)
    k_safe = np.where(k == 0, 1e-10, k)
    T = transfer_BBKS(k_safe, Om_h2)
    return k_safe**ns * T**2

# --- Gaussian random field generator ---
def generate_gaussian_field(N, L, Pk_func, seed=42):
    """Generate a 2D Gaussian random field on an NxN grid of side L.
    Returns the real-space field delta(x,y) and the k-space grid.
    """
    np.random.seed(seed)
    kx = np.fft.fftfreq(N, d=L/N) * 2 * np.pi
    ky = np.fft.fftfreq(N, d=L/N) * 2 * np.pi
    KX, KY = np.meshgrid(kx, ky)
    Pk = Pk_func(KX, KY)
    amplitude = np.sqrt(Pk * (L / N)**2 / 2)
    delta_k = amplitude * (np.random.randn(N, N) + 1j * np.random.randn(N, N))
    delta_k[0, 0] = 0
    delta_x = np.fft.ifft2(delta_k).real
    delta_x /= delta_x.std()  # unit variance before rescaling
    return delta_x, KX, KY

In [ ]:
# ============================================================
# Tutorial: Generate and plot the initial density field
# ============================================================

N = 512      # grid resolution
L = 400.0    # box size in Mpc/h

delta, KX, KY = generate_gaussian_field(N, L, power_spectrum_2d)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(delta, extent=[0, L, 0, L], cmap='RdBu_r',
                vmin=-3, vmax=3, origin='lower')
plt.colorbar(im, ax=ax, label=r'$\delta$')
ax.set_xlabel('$x$ [$h^{-1}$ Mpc]')
ax.set_ylabel('$y$ [$h^{-1}$ Mpc]')
ax.set_title('Initial Gaussian Random Field $\\delta(\\mathbf{x})$')
plt.tight_layout()
plt.show()


This looks like random noise --- no visible structure yet. The Zel'dovich approximation will turn this into the cosmic web by displacing mass elements according to the gravitational potential.

## Part 2 --- Computing the Displacement Field

The Zel'dovich displacement $\boldsymbol{\Psi}$ is related to the gravitational potential $\Phi$ by $\boldsymbol{\Psi} = -\nabla\Phi / (4\pi G \bar\rho a)$. Since the Poisson equation gives $\nabla^2 \Phi = 4\pi G \bar\rho a^2 \delta$, in Fourier space:

$$\tilde\Phi(\mathbf{k}) = -\frac{\tilde\delta(\mathbf{k})}{k^2}$$

The displacement field components are therefore:

$$\tilde\Psi_x = \frac{i k_x\, \tilde\delta}{k^2}, \qquad \tilde\Psi_y = \frac{i k_y\, \tilde\delta}{k^2}$$

We compute these in Fourier space and transform back.

In [ ]:
# ============================================================
# Tutorial: Compute displacement field via FFT
# ============================================================

# The Zel'dovich mapping is: x(q, z) = q + D(z) * Psi(q)
# where Psi is computed from the z=0 linear density field delta_0.
# The initial conditions are set at some high redshift z_init.

# Fourier transform of the density field (representing delta at z=0)
delta_k = np.fft.fft2(delta)

K2 = KX**2 + KY**2
K2[0, 0] = 1

# Displacement: Psi = ik * delta / k^2 (from Poisson equation)
Psi_x_k = 1j * KX * delta_k / K2
Psi_y_k = 1j * KY * delta_k / K2
Psi_x_k[0, 0] = 0
Psi_y_k[0, 0] = 0

Psi_x = np.fft.ifft2(Psi_x_k).real
Psi_y = np.fft.ifft2(Psi_y_k).real

# --- Physical amplitude normalisation ---
# Rescale so that at z=0 (D=1), the RMS displacement is ~5 Mpc/h,
# consistent with a Planck LCDM cosmology in a ~500 Mpc/h box.
# We rescale delta by the same factor (since Psi = ik*delta/k^2).
psi_rms = np.sqrt(Psi_x.var() + Psi_y.var())
target_rms = 5.0  # Mpc/h at z=0 (D=1)
alpha = target_rms / psi_rms
delta *= alpha
Psi_x *= alpha
Psi_y *= alpha

print(f"Physical normalisation: alpha = {alpha:.4f}")
print(f"delta_0 field: sigma = {delta.std():.4f}")
print(f"Displacement: RMS |Psi| = {target_rms:.1f} Mpc/h at z=0")
print(f"  At z=0:  D={growth_factor(0):.2f}, displacement = {target_rms*growth_factor(0):.1f} Mpc/h ({target_rms*growth_factor(0)/(L/N):.1f} cells)")
print(f"  At z=1:  D={growth_factor(1):.2f}, displacement = {target_rms*growth_factor(1):.1f} Mpc/h ({target_rms*growth_factor(1)/(L/N):.1f} cells)")
print(f"  At z=10: D={growth_factor(10):.2f}, displacement = {target_rms*growth_factor(10):.1f} Mpc/h ({target_rms*growth_factor(10)/(L/N):.1f} cells)")

In [ ]:
# ============================================================
# Tutorial: Plot the displacement field as a quiver plot
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))
# Subsample for clarity
s = 16  # plot every 16th arrow
x = np.linspace(0, L, N)
Y, X = np.meshgrid(x, x)
ax.quiver(X[::s, ::s], Y[::s, ::s], Psi_x[::s, ::s], Psi_y[::s, ::s],
          scale=L*1, alpha=1., color='steelblue')
ax.set_xlabel('$x$ [$h^{-1}$ Mpc]')
ax.set_ylabel('$y$ [$h^{-1}$ Mpc]')
ax.set_title('Displacement Field $\\mathbf{\\Psi}(\\mathbf{q})$')
ax.set_xlim(0, L)
ax.set_ylim(0, L)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Part 3 --- Watching the Cosmic Web Form

The Zel'dovich approximation maps initial (Lagrangian) positions $\mathbf{q}$ to final (Eulerian) positions $\mathbf{x}$:

$$\mathbf{x}(\mathbf{q}, t) = \mathbf{q} + D(t)\,\boldsymbol{\Psi}(\mathbf{q})$$

By increasing the growth factor $D$ from 0 (initial conditions) to $\sim 1$ (today), we watch structure form. We deposit particles onto a grid using **Cloud-in-Cell (CIC)** interpolation to visualise the density field at each epoch.

In [ ]:
# ============================================================
# Tutorial: Zel'dovich mapping + CIC density estimation
# ============================================================

def apply_zeldovich(q_x, q_y, Psi_x, Psi_y, D, L):
    """Apply Zel'dovich displacement: x = q + D * Psi, with periodic BC."""
    x = (q_x + D * Psi_x) % L
    y = (q_y + D * Psi_y) % L
    return x, y


def deposit_cic(x, y, N, L):
    """Cloud-in-Cell density assignment onto NxN grid."""
    dx = L / N
    density = np.zeros((N, N))
    # Grid indices
    ix = x / dx
    iy = y / dx
    ix0 = np.floor(ix).astype(int) % N
    iy0 = np.floor(iy).astype(int) % N
    ix1 = (ix0 + 1) % N
    iy1 = (iy0 + 1) % N
    wx = ix - np.floor(ix)
    wy = iy - np.floor(iy)
    # Deposit with np.add.at for correct binning
    np.add.at(density, (ix0, iy0), (1 - wx) * (1 - wy))
    np.add.at(density, (ix1, iy0), wx * (1 - wy))
    np.add.at(density, (ix0, iy1), (1 - wx) * wy)
    np.add.at(density, (ix1, iy1), wx * wy)
    # Normalise to overdensity
    density = density / density.mean() - 1
    return density

In [ ]:
# ============================================================
# Tutorial: Cosmic web at four cosmological epochs
# ============================================================

redshifts = [10, 3, 1, 0]
D_values = [growth_factor(z) for z in redshifts]

# Lagrangian grid
q_x, q_y = np.meshgrid(np.linspace(0, L, N, endpoint=False),
                        np.linspace(0, L, N, endpoint=False))
q_x = q_x.ravel()
q_y = q_y.ravel()
psi_x_flat = Psi_x.ravel()
psi_y_flat = Psi_y.ravel()

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, z, D in zip(axes.ravel(), redshifts, D_values):
    x, y = apply_zeldovich(q_x, q_y, psi_x_flat, psi_y_flat, D, L)
    rho = deposit_cic(x, y, N, L)
    im = ax.imshow(np.log10(2. + rho), extent=[0, L, 0, L], cmap='inferno',
                    origin='lower', vmin=0, vmax=1.)
    ax.set_title(f'$z = {z}$ ($D = {D:.3f}$)', fontsize=14)
    ax.set_xlabel('$x$ [$h^{-1}$ Mpc]')
    ax.set_ylabel('$y$ [$h^{-1}$ Mpc]')
fig.suptitle("Zel'dovich Approximation: Structure Growth from $z=10$ to Today",
             fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:**

- **$z = 10$ ($D \approx 0.08$):** The density field is almost uniform --- fluctuations are deep in the linear regime ($\delta \ll 1$). No visible structure.
- **$z = 3$ ($D \approx 0.25$):** Faint sheet-like structures (pancakes) begin to emerge as matter collapses along one axis.
- **$z = 1$ ($D \approx 0.55$):** A clear filamentary network is forming, with dense nodes at filament intersections. This is the epoch when most galaxy clusters are assembling.
- **$z = 0$ ($D = 1$):** The cosmic web is fully developed: filaments, nodes, and voids. On small scales, the Zel'dovich approximation is breaking down (shell crossing).

The growth factor $D(z)$ from Lecture 5 controls the amplitude at each epoch: $P(k, z) = D^2(z)\, P(k, 0)$ in linear theory.

## Part 4 --- Measuring the Power Spectrum

The Zel'dovich approximation conserves mass but redistributes it.  How does this
change the power spectrum?  We can measure $P(k)$ from the evolved density field
and compare it to the **input** (linear) power spectrum.

On **large scales** ($k \ll k_{\rm NL}$), the Zel'dovich field should reproduce
the linear $P(k)$ because those modes haven't gone nonlinear yet.  On **small
scales** ($k \gg k_{\rm NL}$), mode--mode coupling transfers power from large to
small scales, producing excess power --- this is the beginning of the one-halo
term discussed in the lecture.

We measure $P(k)$ by squaring the Fourier amplitudes and averaging in radial
$k$-bins (the standard estimator for a periodic box).

In [ ]:
# ============================================================
# Tutorial: Measure P(k) and compare to linear prediction
# ============================================================

def measure_pk(density, L, N, n_bins=40):
    """Measure the isotropic power spectrum P(k) from a 2D density field."""
    delta_k = np.fft.fft2(density)
    Pk_2d = np.abs(delta_k)**2 * (L / N)**2 / L**2
    kx = np.fft.fftfreq(N, d=L/N) * 2 * np.pi
    ky = np.fft.fftfreq(N, d=L/N) * 2 * np.pi
    KX_m, KY_m = np.meshgrid(kx, ky)
    K = np.sqrt(KX_m**2 + KY_m**2)
    k_edges = np.linspace(0, kx.max(), n_bins + 1)
    k_centres = 0.5 * (k_edges[:-1] + k_edges[1:])
    Pk_binned = np.zeros(n_bins)
    for b in range(n_bins):
        mask = (K >= k_edges[b]) & (K < k_edges[b+1])
        if mask.sum() > 0:
            Pk_binned[b] = Pk_2d[mask].mean()
    return k_centres, Pk_binned


# --- Use z=50 (deep linear regime) as the reference ---
# This way the CIC window function cancels in the ratio,
# and we're comparing like-with-like.
z_ref = 50
D_ref = growth_factor(z_ref)
x_ref, y_ref = apply_zeldovich(q_x, q_y, psi_x_flat, psi_y_flat, D_ref, L)
rho_ref = deposit_cic(x_ref, y_ref, N, L)
k_ref, Pk_ref = measure_pk(rho_ref, L, N)
valid = (k_ref > 0) & (Pk_ref > 0)

print(f"Reference: z = {z_ref}, D = {D_ref:.5f}")
print(f"At z={z_ref}, everything is deeply linear (delta ~ {delta.std()*D_ref:.1e})")

# --- P(k) at several redshifts ---
z_list = [10, 3, 1, 0]
colors = ['#27ae60', '#2471a3', '#e67e22', '#c0392b']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: P(k) at z=0 vs linear prediction ---
D0 = growth_factor(0)
x0, y0 = apply_zeldovich(q_x, q_y, psi_x_flat, psi_y_flat, D0, L)
rho0 = deposit_cic(x0, y0, N, L)
k0, Pk0 = measure_pk(rho0, L, N)

# Linear prediction scaled from z=50 reference
Pk_linear_z0 = Pk_ref * (D0 / D_ref)**2

ax1.loglog(k_ref[valid], Pk_linear_z0[valid], 'k--', lw=2.5,
           label='Linear: $D^2(0)\\times P(k, z\\!=\\!50)$')
ax1.loglog(k0[valid], Pk0[valid], color='#e67e22', lw=2,
           label="Zel'dovich $z\\!=\\!0$")
ax1.set_xlabel('$k$ [$h$ Mpc$^{-1}$]')
ax1.set_ylabel('$P(k)$')
ax1.set_title('$z = 0$: Linear Prediction vs Zel\'dovich')
ax1.legend(fontsize=12)
ax1.set_xlim(k_ref[valid][0], k_ref[valid][-1])

# --- Right: ratio at several redshifts ---
for z, col in zip(z_list, colors):
    D = growth_factor(z)
    xd, yd = apply_zeldovich(q_x, q_y, psi_x_flat, psi_y_flat, D, L)
    rhod = deposit_cic(xd, yd, N, L)
    kd, Pkd = measure_pk(rhod, L, N)
    # Ratio to linear prediction: P(k,z) / [(D(z)/D_ref)^2 * P(k,z_ref)]
    Pk_lin_pred = Pk_ref #* (D / D_ref)**2
    ratio = np.where(Pk_lin_pred > 0, Pkd / Pk_lin_pred, np.nan)
    ax2.semilogx(kd[valid], ratio[valid], color=col, lw=2,
                  label=f'$z = {z}$ ($D = {D:.3f}$)')

ax2.axhline(1.0, color='k', ls='--', lw=1.5, alpha=0.5, label='Linear prediction')
ax2.set_xlabel('$k$ [$h$ Mpc$^{-1}$]')
ax2.set_ylabel('$P_{\\rm Zel}(k,z) \;/\; [D(z)/D(z_{\\rm ref})]^2\\, P(k,z_{\\rm ref})$')
ax2.set_title(f'Ratio to Linear Theory (reference: $z = {z_ref}$)')
ax2.legend(fontsize=11)
#ax2.set_xlim(k_ref[valid][0], k_ref[valid][-1])
#ax2.set_ylim(0, 3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"  Reference epoch: z={z_ref} (deep linear regime, D={D_ref:.5f})")
print("  Left:  At z=0, Zel'dovich matches linear on large scales,")
print("         excess power on small scales from nonlinear mode coupling")
print("  Right: Ratio ~ 1 on large scales (linear theory works)")
print("         Ratio > 1 on small scales, growing toward z=0")
print("         This excess is the precursor of the one-halo term")

## Exercise: Explore the Dependence on the Power Spectrum

**Tasks:**

1. **Spectral index dependence:** Change the spectral index $n_s$ to 0.5, 1.0, and 1.5. For each value, regenerate the initial Gaussian field and produce the 4-panel cosmic web figure (at $D = 0.2, 0.5, 1.0, 2.0$). How does the *texture* of the cosmic web change with $n_s$?

2. **Exponential cutoff:** Add an exponential cutoff to the power spectrum:
$$P(k) \propto k^{n_s} \exp(-k^2 / k_{\rm cut}^2)$$
with $k_{\rm cut} = 0.2\;{\rm Mpc}^{-1}$. This mimics the effect of free-streaming (e.g., warm dark matter). How does this change the small-scale structure?

**Hints:**
- Modify `power_spectrum_2d` or write a new function with extra parameters.
- Use `generate_gaussian_field(N, L, Pk_func, seed=...)` with different seeds.
- Reuse the `apply_zeldovich` and `deposit_cic` functions from Part 3.

In [ ]:
# ============================================================
# Exercise --- Power spectrum dependence
# ============================================================

# YOUR CODE HERE:
#
# Task 1: Write a function make_pk_func(ns) that returns a power
#          spectrum function P(k) = k^ns. Loop over ns = 0.5, 1.0, 1.5
#          and produce a 4-panel figure for each.
#
# Task 2: Write pk_cutoff(kx, ky, ns=0.9649, k_cut=0.1) that returns
#          k^ns * exp(-k^2 / k_cut^2). Generate the field and make
#          the 4-panel figure.
#
#
# Hints:
# - For each new P(k), you need to regenerate delta, recompute Psi,
#   and redo the Zeldovich mapping.
# - Use the same N, L, D_values, and plotting code from Part 3.
# - np.fft.fft2, np.fft.ifft2 for Fourier transforms.


## Summary

In this practical you have:

- **Generated a 2D Gaussian random field** from the BBKS matter power spectrum $P(k) = k^{n_s}\,T^2(k)$
- **Computed the displacement field** $\boldsymbol{\Psi}$ by solving the Poisson equation in Fourier space
- **Applied the Zel'dovich mapping** $\mathbf{x} = \mathbf{q} + D\,\boldsymbol{\Psi}$ to watch the cosmic web emerge
- **Measured the power spectrum** of the evolved density field and compared it to the linear input --- seeing the onset of nonlinear mode coupling on small scales

### Key takeaways

1. The Zel'dovich approximation is exact to **first order in Lagrangian perturbation theory** and is used to set **initial conditions for N-body simulations**
2. Structure forms hierarchically: pancakes form first, then filaments, then nodes at filament intersections
3. The **power spectrum** of the evolved field matches the linear input on large scales but develops **excess small-scale power** from nonlinear mode coupling --- this is the origin of the one-halo term
4. The spectral index $n_s$ controls the **texture** of the web: higher $n_s$ means more small-scale power


---

*This practical is part of the Cosmology course. The Zel'dovich approximation provides a bridge between linear perturbation theory (Lecture 5) and the nonlinear regime of N-body simulations (Lecture 6).*